# A form generated from a spec

An app on a client: `ess.apps.panel.App` generates a form for each spec from its pydantic models, with a tab per parameter, its widget following from `ess.spec.places.place`, a table edited a row at a time, and an input of its type for each field of a group.
The app submits each request through the client, shows each reason the backend refuses it by its place, and why a record failed.
It picks files from `client.files`, and references from the client's records whose outputs fit the parameter and that the client may read.
Below the form, `ess.apps.panel.Outputs` lists each output of each record: what it is, how it was made, its value in brief, and a plot of an output with an array.

The specs are those of `toy_sans.py`.

In [ ]:
import tempfile
from pathlib import Path

import panel as pn

import toy_sans as t
from ess.apps import local
from ess.apps.panel import App
from ess.apps.testing import FakeFiles

pn.extension('tabulator')

## The app

The client is that of a backend in this process, `local()`, with the bindings of the specs.
Its file source holds the toy runs, each with its role.
Try: find a beam centre in run 101, and a transmission from run 120.
Then the form of I(Q) offers each as a record.
The form of I(Q) of summed runs takes the sample and can runs as tables, with a row per run: pick a run in the editor below the table, and click Add as new row.
A table starts with nothing given, until a row is added or removed.

A submission returns at once, and the record is listed pending.
The client tells no one when a record finishes, so the app polls its status twice a second.
Counts per run takes a second per run, long enough to see a record pending: give it runs 101, 102, and 103, and plot its 2-D output once it has completed.
Select an output to see its record, its spec, the records and files it read, and its params as the record holds them.

The cell shows the app here, and `panel serve spec-form.ipynb --show`, run in this folder, serves it as a page of its own.

In [ ]:
files = FakeFiles('p1', Path(tempfile.mkdtemp()))
for runs, role in (
    (t.SAMPLE_RUNS, 'sample'),
    (t.CAN_RUNS, 'can'),
    (t.TRANSMISSION_RUNS, 'transmission'),
):
    for run, content in runs.items():
        files.measure(run, content, role=role)

bindings = [
    t.find_centre_binding,
    t.transmission_binding,
    t.single,
    t.multi,
    t.forward_binding,
    t.counts_binding,
]
client = local(proposal='p1', files=files, bind=bindings)
app = App(client)
app.panel().servable()